[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indianspeedster/TorchCode/blob/master/templates/06_multihead_attention.ipynb)

# 🔴 Hard: Multi-Head Attention

Implement **Multi-Head Attention** from scratch — the core building block of the Transformer.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(Q W_i^Q,\; K W_i^K,\; V W_i^V)$$

### Signature
```python
class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, Q, K, V) -> torch.Tensor: ...
```

### Requirements
- Use `nn.Linear(d_model, d_model)` for `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`
- `d_k = d_model // num_heads` per head
- `forward(Q, K, V)`: Q is `(B, seq_q, d_model)`, K/V are `(B, seq_k, d_model)`
- Must support **cross-attention** (`seq_q != seq_k`)
- Do **NOT** use `torch.nn.MultiheadAttention`
- You **may** use `torch.softmax` and `torch.matmul`

### Steps
1. Project: `q = self.W_q(Q)`, `k = self.W_k(K)`, `v = self.W_v(V)`
2. Reshape to `(B, num_heads, seq, d_k)`
3. Scaled dot-product attention per head
4. Concat heads → `(B, seq_q, d_model)`
5. Output projection: `self.W_o(concat)`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn
import math

In [12]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int):
        # Initialize W_q, W_k, W_v, W_o
        self.d_model = d_model
        self.num_heads = num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.d_head = d_model // num_heads


    def forward(self, Q, K, V):
        b,q_s,_ = Q.shape
        _,kv_s,_ = V.shape

        q = self.W_q(Q).reshape(b, q_s, self.num_heads, self.d_head).transpose(2,1)
        k = self.W_k(K).reshape(b, kv_s, self.num_heads, self.d_head).transpose(2,1)
        v = self.W_v(V).reshape(b, kv_s, self.num_heads, self.d_head).transpose(2,1)
        score = q@k.transpose(-1,-2)
        score = torch.softmax(score/self.d_head**0.5, dim=-1)
        attn_weight = score@v
        print(attn_weight.shape)
        attn_weight = attn_weight.transpose(2,1).contiguous().view(b,q_s,self.d_model)
        return self.W_o(attn_weight)

In [13]:
# 🧪 Debug
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=32, num_heads=4)
print("W_q type:", type(mha.W_q))          # should be nn.Linear
print("W_q.weight shape:", mha.W_q.weight.shape)  # (32, 32)

x = torch.randn(2, 6, 32)
out = mha.forward(x, x, x)
print("Output shape:", out.shape)          # (2, 6, 32)

# Cross-attention
Q = torch.randn(1, 3, 32)
K = torch.randn(1, 7, 32)
V = torch.randn(1, 7, 32)
out2 = mha.forward(Q, K, V)
print("Cross-attn shape:", out2.shape)     # (1, 3, 32)

W_q type: <class 'torch.nn.modules.linear.Linear'>
W_q.weight shape: torch.Size([32, 32])
torch.Size([2, 4, 6, 8])
Output shape: torch.Size([2, 6, 32])
torch.Size([1, 4, 3, 8])
Cross-attn shape: torch.Size([1, 3, 32])


In [14]:
# ✅ SUBMIT
from torch_judge import check
check("mha")


🧪 Testing: Multi-Head Attention (Hard)
──────────────────────────────────────────────────
torch.Size([2, 4, 6, 8])
  ✅ [1/6] Output shape (4.5ms)
  ✅ [2/6] Uses nn.Linear with correct shapes (0.6ms)
torch.Size([1, 2, 4, 8])
  ✅ [3/6] Numerical correctness vs reference (2.6ms)
torch.Size([1, 2, 4, 8])
  ✅ [4/6] Gradient flow (2.9ms)
torch.Size([1, 4, 3, 8])
  ✅ [5/6] Cross-attention (seq_q != seq_k) (0.7ms)
  ✅ [6/6] Different heads give different outputs (2.0ms)
──────────────────────────────────────────────────
  🎉 All 6 tests passed! (13.3ms total)
  Progress saved. Run status() to see your dashboard.

